In [6]:
# 1. MOUNT GOOGLE DRIVE & EXTRACT DATA LOCALLY
from google.colab import drive
import os

# Mount Drive (this will ask for permission)
drive.mount('/content/drive')

# --- UPDATE THESE PATHS ---
ZIP_PATH = '/content/drive/MyDrive/Proyecto_ML/x_train.zip'
CSV_PATH = '/content/drive/MyDrive/Proyecto_ML/y_train_v2.csv'
# --------------------------

# Define local path for fast I/O during training
LOCAL_DATA_DIR = '/content/data_local'
IMG_FOLDER = os.path.join(LOCAL_DATA_DIR, 'x_train')

# Unzip data locally if it hasn't been unzipped yet
if not os.path.exists(IMG_FOLDER):
    print("Extracting images locally for faster processing. This might take a minute...")
    # The '!' runs a shell command directly from Python in Colab
    os.system(f"unzip -q {ZIP_PATH} -d {LOCAL_DATA_DIR}")
    print("Extraction complete!")
else:
    print("Images already extracted locally.")




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting images locally for faster processing. This might take a minute...
Extraction complete!


In [ ]:
# 2. DATA PREPROCESSING AND BASELINE TRAINING
import pandas as pd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, f1_score, classification_report

def load_data(csv_path, img_folder, img_size=(64, 64)):
    """
    Loads PNG images and their corresponding labels from the CSV.
    Resizes images to img_size and normalizes the pixel values.
    """
    labels_df = pd.read_csv(csv_path)
    images = []
    labels = []

    for index, row in labels_df.iterrows():
        img_id = row['id']
        label = row['target']
        img_path = os.path.join(img_folder, f"{img_id}.png")

        # Read image in grayscale
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is not None:
            # Resize and normalize [0, 1]
            img = cv2.resize(img, img_size)
            img = img / 255.0
            images.append(img)
            labels.append(label)

    return np.array(images), np.array(labels)

print("\nLoading and preprocessing data...")
X, y = load_data(CSV_PATH, IMG_FOLDER)

# Flatten images for classical ML (SVM requires 1D features)
X_flattened = X.reshape(X.shape[0], -1)

# Split 80/20 with stratification to maintain class distribution
X_train, X_val, y_train, y_val = train_test_split(
    X_flattened, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")

print("\nApplying PCA for dimensionality reduction...")
# Keep 95% of variance to reduce noise and speed up SVM
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

print("\nTraining SVM Baseline model (RBF Kernel)...")
svm_model = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_model.fit(X_train_pca, y_train)

# 3. EVALUATION
print("\nEvaluating the model...")
y_pred = svm_model.predict(X_val_pca)

f1 = f1_score(y_val, y_pred, average='macro')
conf_matrix = confusion_matrix(y_val, y_pred)

print(f"\n--- RESULTS ---")
print(f"Validation F1-Score (Macro): {f1:.4f}")
print("Confusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", classification_report(y_val, y_pred))